# Lecture 9: Applied Regression in Economics
## Credibility, Diagnostics, and Knowing When to Worry

**BANA 4373 — Dr. Fidel Gonzalez — Sam Houston State University**

---

### About today's data

We are using **calibrated synthetic data** — not purely made up, not real.
The parameters we planted into the data-generating process (DGP) mirror published labor economics estimates:

| Parameter | Planted value | Real-world source |
|---|---|---|
| Return to education | ~7.5% per year | CPS/Mincer estimates |
| Union wage premium | ~10% | Card (1996) |
| Raw gender gap | ~7% | IPUMS CPS |
| Returns to experience | Diminishing (quadratic) | Mincer (1974) |
| Ability confound | Correlated with education | Griliches (1977) |

**Why synthetic?** In real data you never know the true coefficient. Today we do. That is the point. The homework asks you to replicate this on real CPS data.

---

### Three-act structure

| Act | Focus | Goal |
|---|---|---|
| **I — Explore & Commit** | EDA, functional form choice | Form and write down expectations before running models |
| **II — Break It** | Subsample splits, OVB reveal, diagnostics | Discover where the regression fails and why |
| **III — Report Honestly** | Coefficient stability, executive summary | Write a defensible 3-sentence brief |

You will maintain a **living executive summary** — one sentence updated at key moments — that becomes your final deliverable and the class participation for today. 

Type your name here since, you will be submitting this for participation grade. Make sure when you sbumit that you have run all the code and answer all the questions:

Student Name: 

## 0) Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import statsmodels.api as sm
import statsmodels.formula.api as smf
from scipy import stats

np.random.seed(4370)
pd.set_option('display.max_columns', 50)
pd.set_option('display.float_format', '{:.4f}'.format)

# Plotting defaults
plt.rcParams['figure.figsize'] = (9, 4)
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False

print('Ready.')

---
## ACT I — Explore and Commit
### 1) Build the calibrated synthetic dataset

The DGP below encodes the structure we discussed in the slides.
Read it carefully — you are looking at the ground truth that applied economists can never see in real data.

In [ ]:
n = 1500

# ----- UNOBSERVED CONFOUNDER -----
# Ability is correlated with education (key to the OVB story).
# In real data this column does not exist.
ability = np.random.normal(0, 1, n)

# ----- OBSERVABLE VARIABLES -----
educ   = np.clip(np.round(12 + 2.2*ability + np.random.normal(0, 2.0, n)), 8, 20).astype(int)
exper  = np.clip(np.round(np.random.uniform(0, 35, n)), 0, 35).astype(int)
female = np.random.binomial(1, 0.50, n)
union  = np.random.binomial(1, 0.18 + 0.02*(educ >= 14), n)  # slightly more likely w/ college

# ----- TRUE LOG-WAGE PROCESS -----
# These are the planted coefficients. Keep this cell in mind during the OVB demo.
log_wage = (
    1.60
    + 0.075 * educ                      # ~7.5% return to education
    + 0.035 * exper                      # positive returns to experience
    - 0.00055 * (exper**2)               # ... but diminishing
    + 0.10  * union                      # ~10% union premium
    - 0.07  * female                     # ~7% raw gender gap
    + 0.18  * ability                    # OMITTED VARIABLE (unobserved in real life)
    + np.random.normal(0, 0.25, n)       # noise
)

wage = np.exp(log_wage)

df = pd.DataFrame({
    'wage':     wage,
    'log_wage': np.log(wage),
    'educ':     educ,
    'exper':    exper,
    'female':   female,
    'union':    union,
    'ability':  ability   # pretend you cannot see this column in real life!
})

print(f'Dataset: {len(df):,} observations, {df.shape[1]} variables')
df.head()

### 2) Write your priors (do this BEFORE running any regressions)

This is a professional habit, not busywork. Applied economists write down what they expect before looking at results — it forces you to think about the economics, not just the output.

**📝 Double-click this cell and fill in your expectations:**

| Coefficient | Expected sign | Expected rough magnitude | Reasoning |
|---|---|---|---|
| Education | | | |
| Experience | | | |
| Experience² | | | |
| Union | | | |
| Female | | | |

### 3) EDA — look before you model

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 4))

axes[0].hist(df['wage'], bins=50, edgecolor='white', linewidth=0.3)
axes[0].set_title('Wage (levels)', fontsize=11)
axes[0].set_xlabel('wage ($)')

axes[1].hist(df['log_wage'], bins=50, color='steelblue', edgecolor='white', linewidth=0.3)
axes[1].set_title('Log wage', fontsize=11)
axes[1].set_xlabel('log(wage)')

axes[2].scatter(df['educ'], df['log_wage'], alpha=0.15, s=8, color='steelblue')
# Add a lowess smoother
lowess = sm.nonparametric.lowess(df['log_wage'], df['educ'], frac=0.4)
axes[2].plot(lowess[:, 0], lowess[:, 1], color='red', linewidth=2, label='LOWESS')
axes[2].set_title('Log wage vs. Education', fontsize=11)
axes[2].set_xlabel('years of education')
axes[2].set_ylabel('log(wage)')
axes[2].legend()

plt.tight_layout()
plt.show()

print('\nSummary stats:')
df[['wage', 'log_wage', 'educ', 'exper', 'union', 'female']].describe().T

**📝 EDA checkpoint — update your living summary (Sentence 1 draft):**

> Based on the distributions, I will model _______ as the dependent variable because _______. I expect the relationship with education to look _______ based on the scatter plot.

### 4) Functional form comparison

Run all three standard forms. Compare the $R^2$ and residual patterns, not just the coefficients.

In [ ]:
# Model 1: Level-Level
m1 = smf.ols('wage ~ educ + exper', data=df).fit(cov_type='HC1')

# Model 2: Log-Level (standard in labor economics)
m2 = smf.ols('log_wage ~ educ + exper', data=df).fit(cov_type='HC1')

# Model 3: Log-Level with quadratic experience (Mincer specification)
m3 = smf.ols('log_wage ~ educ + exper + I(exper**2)', data=df).fit(cov_type='HC1')

# Quick comparison table
comparison = pd.DataFrame({
    'Model':    ['m1: level-level', 'm2: log-level', 'm3: log-level + exp²'],
    'Dep var':  ['wage', 'log_wage', 'log_wage'],
    'R²':       [m1.rsquared, m2.rsquared, m3.rsquared],
    'AIC':      [m1.aic,      m2.aic,      m3.aic],
    'educ_coef':[m1.params['educ'], m2.params['educ'], m3.params['educ']],
    'educ_se':  [m1.bse['educ'],    m2.bse['educ'],    m3.bse['educ']],
})
comparison.set_index('Model').round(4)

In [ ]:
# Interpret the education coefficient in m2 precisely
b = m2.params['educ']
pct_approx  = 100 * b
pct_precise = 100 * (np.exp(b) - 1)

print(f'Education coefficient (m2): {b:.4f}')
print(f'  Rule-of-thumb (100*b):    {pct_approx:.2f}%')
print(f'  Precise (100*(exp(b)-1)): {pct_precise:.2f}%')
print(f'  Difference:               {abs(pct_approx - pct_precise):.3f} ppts  ← matters when |b|>0.1')
print()
print(f'TRUE planted value: 7.50%')
print(f'Recovered estimate: {pct_precise:.2f}%  (biased upward — we are still omitting ability)')

---
## ACT II — Break the Regression
### 5) Robust vs. conventional SE — does it change your conclusions?

The key question is not "are the SE different" but "does the difference change whether I reject the null".

In [ ]:
# Full model: log-level, all controls, both SE variants
m_conv = smf.ols('log_wage ~ educ + exper + I(exper**2) + union + female', data=df).fit()
m_rob  = smf.ols('log_wage ~ educ + exper + I(exper**2) + union + female', data=df).fit(cov_type='HC1')

# Side-by-side SE and p-values
se_compare = pd.DataFrame({
    'Coef':      m_conv.params,
    'SE (OLS)':  m_conv.bse,
    'SE (HC1)':  m_rob.bse,
    'SE ratio':  m_rob.bse / m_conv.bse,
    'p (OLS)':   m_conv.pvalues,
    'p (HC1)':   m_rob.pvalues,
    'Conclusion changes?': (m_conv.pvalues < 0.05) != (m_rob.pvalues < 0.05)
}).round(4)

se_compare

In [ ]:
# Visual: residuals vs fitted — check for heteroskedasticity
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for ax, model, title in zip(axes,
                             [m_conv, m_rob],
                             ['Residuals vs Fitted (conventional SE)',
                              'Residuals vs Fitted (robust SE fit is identical)']):
    ax.scatter(model.fittedvalues, model.resid, alpha=0.15, s=8)
    ax.axhline(0, color='red', linewidth=1)
    # Add lowess smoother to residuals to reveal patterns
    lw = sm.nonparametric.lowess(model.resid, model.fittedvalues, frac=0.3)
    ax.plot(lw[:, 0], lw[:, 1], color='orange', linewidth=2, label='LOWESS trend')
    ax.set_title(title, fontsize=10)
    ax.set_xlabel('Fitted values')
    ax.set_ylabel('Residuals')
    ax.legend()

plt.tight_layout()
plt.show()
print('Note: the two plots are identical because coefficients do not change under robust SE.')
print('Only the standard errors (and confidence intervals) differ.')

### 6) Exercise A: Subsample stability ("Break the Regression")

Run the full log-wage model separately for:
- **Low-education workers** (`educ <= 12`)
- **High-education workers** (`educ >= 16`)

**Before running:** predict whether the `female` coefficient will be larger, smaller, or the same across groups. Write it in the cell below.

**Hint for after:** check union rates by education × gender to help explain what you find.

In [ ]:
# ----- YOUR PREDICTION (fill in before running) -----
# In the low-education subsample, I expect the female coefficient to be: ___
# Because: ___
# In the high-education subsample, I expect the female coefficient to be: ___
# Because: ___

# ----- YOUR CODE HERE -----
df_low  = df[df['educ'] <= 12].copy()
df_high = df[df['educ'] >= 16].copy()

# Hint: use smf.ols with cov_type='HC1'
# Then compare the female coefficient across the two models


In [ ]:
# ========================================================
# ANSWER KEY — run only after attempting above
# ========================================================

m_low  = smf.ols('log_wage ~ educ + exper + I(exper**2) + union + female', data=df_low ).fit(cov_type='HC1')
m_high = smf.ols('log_wage ~ educ + exper + I(exper**2) + union + female', data=df_high).fit(cov_type='HC1')

result = pd.DataFrame({
    'n obs':           [len(df_low),              len(df_high)],
    'female_coef':     [m_low.params['female'],    m_high.params['female']],
    'female_se':       [m_low.bse['female'],        m_high.bse['female']],
    'union_rate':      [df_low['union'].mean(),    df_high['union'].mean()],
    'union_rate_F':    [df_low.loc[df_low['female']==1, 'union'].mean(),
                        df_high.loc[df_high['female']==1, 'union'].mean()],
}, index=['Low-educ (≤12)', 'High-educ (≥16)'])

print('Female coefficient by education subgroup:')
print(result.round(4))
print()
print('Interpretation prompt:')
print('  Why might the gender gap look different across education groups?')
print('  Is this a problem with the data, or does it reflect real heterogeneity?')
print('  What does this imply for a single pooled regression?')

### 7) Exercise B: OVB — vote before you run

We have been omitting `ability` from all models so far (as you would in real data — it is unobserved).

**Before running the next cell:**

1. Will the education coefficient go **up**, **down**, or **stay the same** when we add `ability`?
2. By roughly how much (as a % of the current estimate)?
3. Which direction does the OVB formula predict? (Hint: what is the sign of `ability → wages`? What is the sign of `ability → education`?)

**📝 Write your vote here before running:**

> My prediction: the education coefficient will go _______ by about _______% because _______.

In [ ]:
# OVB demonstration: progressively add controls, watch the education coefficient

models = {
    'm2: educ + exper':                    smf.ols('log_wage ~ educ + exper', data=df).fit(cov_type='HC1'),
    'm3: + exp² + union + female':         smf.ols('log_wage ~ educ + exper + I(exper**2) + union + female', data=df).fit(cov_type='HC1'),
    'm4: + ability (UNOBSERVED in reality)': smf.ols('log_wage ~ educ + exper + I(exper**2) + union + female + ability', data=df).fit(cov_type='HC1'),
}

ovb_table = pd.DataFrame([
    {
        'Model': name,
        'educ coef':      m.params['educ'],
        'educ SE (HC1)':  m.bse['educ'],
        '95% CI lower':   m.conf_int().loc['educ', 0],
        '95% CI upper':   m.conf_int().loc['educ', 1],
        'R²':             m.rsquared,
    }
    for name, m in models.items()
])

print('Education coefficient across models (TRUE value = 0.075):')
ovb_table.set_index('Model').round(4)

In [ ]:
# Quantify the bias formally
m_without = models['m2: educ + exper']
m_with    = models['m4: + ability (UNOBSERVED in reality)']

# OVB = coef without - coef with
ovb = m_without.params['educ'] - m_with.params['educ']

# Auxiliary regression: ability on education (to verify the formula)
delta = smf.ols('ability ~ educ', data=df).fit().params['educ']
beta2 = m_with.params['ability']
predicted_bias = beta2 * delta

true_value = 0.075

print('OVB formula verification:')
print(f'  Observed bias (coef without - coef with): {ovb:.4f}')
print(f'  Predicted by formula (β₂ × δ₁):         {predicted_bias:.4f}')
print(f'  Match? {abs(ovb - predicted_bias) < 0.001}')
print()
print(f'Bias as % of true coefficient: {ovb/true_value*100:.1f}%')
print()
print('This is why the twins studies find lower returns to education:')
print('Differencing within twin pairs removes the ability confound.')

### 8) Diagnostics — build the habit

In [ ]:
# Use m_rob (full model, robust SE) as baseline
m_base = m_rob

fig, axes = plt.subplots(1, 3, figsize=(14, 4))

# --- Plot 1: Residuals vs Fitted ---
fitted = m_base.fittedvalues
resid  = m_base.resid
axes[0].scatter(fitted, resid, alpha=0.15, s=8)
axes[0].axhline(0, color='red', linewidth=1)
lw = sm.nonparametric.lowess(resid, fitted, frac=0.3)
axes[0].plot(lw[:, 0], lw[:, 1], color='orange', linewidth=2)
axes[0].set_title('Residuals vs Fitted', fontsize=10)
axes[0].set_xlabel('Fitted values')
axes[0].set_ylabel('Residuals')

# --- Plot 2: Scale-Location (spread of residuals) ---
sqrt_abs_resid = np.sqrt(np.abs(resid))
axes[1].scatter(fitted, sqrt_abs_resid, alpha=0.15, s=8, color='steelblue')
lw2 = sm.nonparametric.lowess(sqrt_abs_resid, fitted, frac=0.3)
axes[1].plot(lw2[:, 0], lw2[:, 1], color='orange', linewidth=2)
axes[1].set_title('Scale-Location', fontsize=10)
axes[1].set_xlabel('Fitted values')
axes[1].set_ylabel('√|Residuals|')

# --- Plot 3: Cook's Distance ---
infl  = m_base.get_influence()
cooks = infl.cooks_distance[0]
axes[2].stem(np.arange(len(cooks)), cooks,
             markerfmt=',', basefmt=' ', linefmt='grey')
axes[2].axhline(4/len(df), color='red', linewidth=1.5, linestyle='--',
                label=f'4/n = {4/len(df):.4f} (common cutoff)')
axes[2].set_title("Cook's Distance", fontsize=10)
axes[2].set_xlabel('Observation index')
axes[2].set_ylabel("Cook's D")
axes[2].legend(fontsize=8)

plt.tight_layout()
plt.show()

# Flag influential observations
n = len(df)
cutoff  = 4 / n
n_infl  = (cooks > cutoff).sum()
print(f'Observations above Cook\'s D cutoff (4/n={cutoff:.4f}): {n_infl} ({n_infl/n*100:.1f}%)')

In [ ]:
# Show top 10 most influential observations
top_idx = np.argsort(cooks)[-10:][::-1]
(
    df.loc[top_idx, ['wage', 'educ', 'exper', 'union', 'female']]
    .assign(log_wage=df.loc[top_idx, 'log_wage'].values,
            cooks_D=cooks[top_idx],
            fitted=m_base.fittedvalues.iloc[top_idx].values,
            residual=m_base.resid.iloc[top_idx].values)
    .round(3)
)

**📝 Diagnostic interpretation prompt:**

Look at the three plots and answer:
1. Does the residual plot suggest nonlinearity (a curve pattern) or heteroskedasticity (a fan pattern)?
2. Is the scale-location plot flat (good) or trending upward (suggests heteroskedasticity)?
3. Are the influential observations economically unusual, or do they look like normal data points?

> **Your interpretation here:** ___

---
## ACT III — Report Honestly
### 9) Coefficient stability plot

This is a standard applied robustness check: plot the education coefficient with 95% CI across nested models. A stable coefficient is evidence that your result is not driven by observable confounders.

In [ ]:
# Define nested model sequence
model_specs = [
    ('Baseline: educ only',                  'log_wage ~ educ'),
    ('+ experience',                         'log_wage ~ educ + exper'),
    ('+ exp² (Mincer)',                      'log_wage ~ educ + exper + I(exper**2)'),
    ('+ union',                              'log_wage ~ educ + exper + I(exper**2) + union'),
    ('+ female',                             'log_wage ~ educ + exper + I(exper**2) + union + female'),
    ('+ ability (unobserved in real data)',  'log_wage ~ educ + exper + I(exper**2) + union + female + ability'),
]

results = []
for label, formula in model_specs:
    m = smf.ols(formula, data=df).fit(cov_type='HC1')
    ci = m.conf_int().loc['educ']
    results.append({
        'Model': label,
        'coef':  m.params['educ'],
        'lower': ci[0],
        'upper': ci[1],
    })

stab = pd.DataFrame(results)

# --- Plot ---
fig, ax = plt.subplots(figsize=(10, 5))

y_pos = range(len(stab))

ax.errorbar(stab['coef'], y_pos,
            xerr=[stab['coef'] - stab['lower'], stab['upper'] - stab['coef']],
            fmt='o', color='steelblue', ecolor='steelblue',
            capsize=4, markersize=7, linewidth=1.5)

ax.axvline(0.075, color='red', linestyle='--', linewidth=1.5, label='True value (0.075)')
ax.axvline(0,     color='gray', linestyle=':',  linewidth=1,   label='Zero')

ax.set_yticks(y_pos)
ax.set_yticklabels(stab['Model'], fontsize=9)
ax.set_xlabel('Education coefficient (log-wage model, HC1 robust 95% CI)', fontsize=10)
ax.set_title('Coefficient Stability Plot: Return to Education', fontsize=12)
ax.legend(fontsize=9)
ax.invert_yaxis()

plt.tight_layout()
plt.show()

print('\nObservable controls: coefficient is relatively stable')
print('Adding unobservable ability: coefficient drops by', round(stab.iloc[-2]['coef'] - stab.iloc[-1]['coef'], 3))
print('This is the OVB we cannot remove in real data.')

### 10) Final executive summary

Write **3 sentences** as if briefing a policy analyst or manager.

**Requirements:**
1. **Finding** — state the main relationship with magnitude, units, and confidence interval.
2. **Uncertainty** — state whether the result is causal or associational; note what robust SE did or did not change.
3. **Limitation** — name the most important threat to validity, ideally quantified.

**Grading rubric:** Would a non-economist understand what you found? Is the uncertainty honest? Is the limitation specific enough to be actionable?

**📝 Executive Summary (replace with your own writing):**

1. [Finding — magnitude, units, CI]

2. [Uncertainty — causal vs. associational; effect of robust SE]

3. [Limitation — the most important threat, quantified if possible]

---

**Example answer** (do not copy; write your own):

> *An additional year of schooling is associated with approximately a 7.0% increase in wages, holding experience and union status constant (robust 95% CI: 6.3–7.7%). This estimate is associational rather than causal; using robust standard errors did not materially change the inference, though the CIs widened slightly for experience. The most important limitation is unobserved ability: when ability is controlled directly, the education coefficient falls to roughly 5.0%, suggesting the baseline estimate is upward biased by about 28% due to the ability-education correlation.*

---
## Extension: If you finish early

Pick one or more of the following:

In [ ]:
# Extension A: Experience profile plot
# At what years of experience does the wage premium peak, according to m3?
# Hint: set the derivative of (b_exper*exper + b_exper2*exper^2) to zero and solve.

m_ext = smf.ols('log_wage ~ educ + exper + I(exper**2) + union + female', data=df).fit(cov_type='HC1')

b_exp  = m_ext.params['exper']
b_exp2 = m_ext.params['I(exper ** 2)']

peak_exp = -b_exp / (2 * b_exp2)
print(f'Peak experience (analytical): {peak_exp:.1f} years')

# Plot the experience profile
exp_range = np.linspace(0, 40, 200)
wage_profile = b_exp * exp_range + b_exp2 * exp_range**2

plt.figure(figsize=(8, 4))
plt.plot(exp_range, wage_profile, color='steelblue', linewidth=2)
plt.axvline(peak_exp, color='red', linestyle='--', label=f'Peak at {peak_exp:.1f} years')
plt.xlabel('Years of experience')
plt.ylabel('Partial effect on log(wage)')
plt.title('Experience-wage profile (Mincer quadratic)')
plt.legend()
plt.show()

In [ ]:
# Extension B: Model comparison with AIC/BIC
specs = [
    'log_wage ~ educ + exper',
    'log_wage ~ educ + exper + I(exper**2)',
    'log_wage ~ educ + exper + I(exper**2) + union + female',
    'log_wage ~ educ + I(educ**2) + exper + I(exper**2) + union + female',
]

ic_results = []
for spec in specs:
    m = smf.ols(spec, data=df).fit()
    ic_results.append({'Formula': spec, 'AIC': m.aic, 'BIC': m.bic, 'R²': m.rsquared})

pd.DataFrame(ic_results).set_index('Formula').round(1)

---
### Additional work

Replicate today's log-wage model on real CPS microdata (IPUMS extract — instructions on GitHub).

**Deliverable:** a stability plot for the education coefficient, plus a 3-sentence brief. Post to GitHub before next class.

**Question to answer:** Do the planted coefficients (7.5% education return, 10% union premium) hold in real data? If they differ, what is your proposed explanation?